# Ridge Regression with California Housing Data

This project uses multiple input features and L2 regularization to predict median house values.


In [ ]:
import pandas as pd
import numpy as np

# 1. Load the California housing dataset and choose the input features.
url = "https://github.com/akmand/datasets/raw/refs/heads/main/california_housing.csv"
df = pd.read_csv(url)
columns = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]

# 2. X contains the input features; y is the median house value we want to predict.
X = df[columns].to_numpy(dtype=float)
y = df["median_house_value"].to_numpy(dtype=float)

# 3. Initialize one weight for each feature and one bias.
w = np.zeros(X.shape[1])
b = 0.0

# 4. Calculate the squared-error cost with an L2 regularization penalty.
def CostF(X,y,w,b,lambda_reg):
  m=len(y)
  error = X@w+b-y
  cost= np.sum(error**2)/(2*m)+np.sum(w**2)*(lambda_reg/(2*m))
  return cost

# 5. Calculate the gradients and update weights and bias with gradient descent.
def Gradient(X,y,w,b,alpha,num_iters,lambda_reg):
  m=len(y)
  for i in range(num_iters):
    error = X@w+b-y
    dj_dw = X.T@(error)/m+lambda_reg/m*w
    dj_db = np.sum(error)/m
    w = w-alpha*dj_dw
    b = b-alpha*dj_db
  return w,b

# 6. Calculate scaling values from the training data and apply them to both sets.
def Fscale(X_train,X_test):
  mu = np.mean(X_train,axis=0)
  sigma = np.std(X_train,axis=0)
  X_train_scaled = (X_train - mu)/sigma
  X_test_scaled = (X_test - mu)/sigma
  return X_train_scaled,X_test_scaled

# 7. Shuffle the dataset and split it into 80% training and 20% testing data.
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
split = int(0.8*len(X))
train = indices[:split]
test = indices[split:]

# Some data in the total_bedrooms column was missing.
# I used AI to help solve this problem because I didn't have
# the necessary skills to handle missing data at the time.
#so what is under me , the train x and test x and idx is from AI not me
train_X = X[train].copy()
test_X = X[test].copy()

# 8. Fill missing bedroom counts using the training-set median for both sets.
idx = columns.index("total_bedrooms")
median = np.nanmedian(train_X[:, idx])

train_X[np.isnan(train_X[:, idx]), idx] = median
test_X[np.isnan(test_X[:, idx]), idx] = median

#now the rest is from me
train_y = y[train]
test_y = y[test]

# 9. Scale the features and target, train the model, and compare train/test cost.
train_X_scaled, test_X_scaled = Fscale(train_X, test_X)
train_y_scaled = train_y / 100000
test_y_scaled = test_y / 100000
N_w,N_b = Gradient(train_X_scaled,train_y_scaled,w,b,alpha=0.05,num_iters=500,lambda_reg=1.0)
print(f"so the trained w is: {N_w} , and the new b is: {N_b}")

print(f"so the prediction of the testing data is: {N_w@test_X_scaled.T+N_b}")
print(f"and the cost func or the accurate of the model in the trained data is: {CostF(train_X_scaled,train_y_scaled,N_w,N_b,lambda_reg=1.0)}")
print("-"*50)
print(f"and the cost func or the accurate of the model in the tested data is: {CostF(test_X_scaled,test_y_scaled,N_w,N_b,lambda_reg=1.0)}")